In [1]:
# imports and settings

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, sosfiltfilt, welch, firwin, lfilter, hilbert
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import write_image

import utils as ut
from reference_detector.ship_noise_pkl_model import run_model as run_reference_model

# do not show warnings
warnings.filterwarnings("ignore")

# autoreload for development
%load_ext autoreload
%autoreload 2

print("Imports complete.")

Imports complete.
Settings: height=800, width=1400, font_size=16
Imports complete.


In [2]:
folder_path = '../data/croatia/2507_1_1000m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

data = []
for file in file_list:
    # print(f"Loading file: {file}")
    _data, fs = librosa.load(os.path.join(folder_path, file))
    data.append(_data)
data = np.concatenate(data)
# data = data[80*fs:]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")

data final shape: (119070000,) of duration: 5400.0 seconds


In [3]:
gpx_file = os.path.join(folder_path, '1000m_route.gpx')
gpx_data = gpxpy.parse(open(gpx_file))

# get the first track
track = gpx_data.tracks[0]

# get the first segment
segment = track.segments[0]

rx_location = gpxpy.gpx.GPXWaypoint(44.327910, 14.700204)

distances = []
for point in segment.points:
    distance = rx_location.distance_2d(point)
    distances.append(distance)

distances = np.array(distances)
t_distances = np.linspace(0, len(data)/fs, len(distances))
f_gps_samples = len(distances) / (len(data)/fs/60)
dive_start, dive_end = int(18*f_gps_samples), int(75*f_gps_samples)

In [4]:
def fft_bandpass(x, fs, f_center, bandwidth=5.0):
    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(len(x), d=1/fs)

    mask = np.logical_and(
        freqs >= f_center - bandwidth / 2,
        freqs <= f_center + bandwidth / 2
    )

    X_filtered = np.zeros_like(X)
    X_filtered[mask] = X[mask]

    return np.fft.irfft(X_filtered, n=len(x))

def bandpass_filter(x, fs, f_center, bandwidth=5.0, order=4):
    x = np.asarray(x)

    if not np.all(np.isfinite(x)):
        raise ValueError("Input signal contains NaN or Inf")

    f_low = f_center - bandwidth / 2
    f_high = f_center + bandwidth / 2

    if f_low <= 0 or f_high >= fs / 2:
        raise ValueError("Invalid band edges")

    sos = butter(
        order,
        [f_low, f_high],
        btype='bandpass',
        fs=fs,
        output='sos'
    )

    return sosfiltfilt(sos, x)

def analytic_signal_phase(x, fs, f, band_width):
    filtered_x = bandpass_filter(x, fs, f_center=f, bandwidth=band_width)
    analytic = hilbert(filtered_x)
    phase = np.angle(analytic)          # wrapped phase
    # phase_unwrapped = np.unwrap(phase)  # continuous phase
    return phase

def band_phase_complex_demod(x, fs, f0, BW, numtaps=401):
    """
    Estimate per-sample phase of a narrow band around f0 using complex demodulation.

    Parameters
    ----------
    x : ndarray
        Real-valued signal
    fs : float
        Sampling frequency (Hz)
    f0 : float
        Center frequency (Hz)
    BW : float
        Bandwidth (Hz)
    numtaps : int
        FIR filter length (odd recommended)

    Returns
    -------
    z : ndarray
        Complex baseband signal
    phase : ndarray
        Unwrapped phase (rad)
    amplitude : ndarray
        Instantaneous amplitude
    """
    n = np.arange(len(x))

    # 1. Complex demodulation
    x_demod = x * np.exp(-1j * 2*np.pi * f0 * n / fs)

    # 2. Low-pass FIR filter
    fc = BW / 2
    h = firwin(numtaps, fc / (fs / 2), window="hann")
    z = lfilter(h, 1.0, x_demod)

    # 3. Phase & amplitude
    phase = np.unwrap(np.angle(z))
    amplitude = np.abs(z)

    # return z, phase, amplitude
    return phase


In [5]:
nperseg = 1000
overlap = 0.5
window = 'hanning'
dc = 100
crop_freq = 8000
norm_size = 5
s2g_quantization_levels = 5
s2g_mode = 'wasserstein'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

welch_detector = ut.WelchDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
talmon_detector = ut.TalmonDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=s2g_quantization_levels)

In [ ]:
# sanity - check spectral power + K only for f0 = 631, 859
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f0_1_bd_score = []
f0_2_bd_score = []

f0_1_s2g_scores = []
f0_2_s2g_scores = []

f0_1_td_scores = []
f0_2_td_scores = []

f0_1 = 631  # Hz
f0_2 = 851  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    pxx, F_pxx = welch_detector.get_feature_vector(_data)
    f0_1_pxx = pxx[np.where(F_pxx >= f0_1)[0][0]]
    f0_2_pxx = pxx[np.where(F_pxx >= f0_2)[0][0]]
    f0_1_bd_score.append(f0_1_pxx)
    f0_2_bd_score.append(f0_2_pxx)

    K, F_K = s2g_detector.get_feature_vector(_data)
    f0_1_K_pxx = K[np.where(F_K >= f0_1)[0][0]]
    f0_2_K_pxx = K[np.where(F_K >= f0_2)[0][0]]
    f0_1_s2g_scores.append(f0_1_K_pxx)
    f0_2_s2g_scores.append(f0_2_K_pxx)

    td_scores, td_F = talmon_detector.get_feature_vector(_data)
    f0_1_td_pxx = td_scores[np.where(td_F >= f0_1)[0][0]]
    f0_2_td_pxx = td_scores[np.where(td_F >= f0_2)[0][0]]
    f0_1_td_scores.append(f0_1_td_pxx)
    f0_2_td_scores.append(f0_2_td_pxx)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")


completed all detectors


In [7]:
fig = make_subplots(rows=5, 
                    cols=1, 
                    row_heights=[0.6, 0.1, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "Welch", "Talmon", "S2G"])

fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=t_distances[dive_start:dive_end], y=distances[dive_start:dive_end], name='Distance', marker_color='red', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_bd_score, name=f'Welch f={f0_1} Hz', marker_color='blue'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_bd_score, name=f'Welch f={f0_2} Hz', marker_color='cyan'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores, name=f'Talmon f={f0_1} Hz', marker_color='blue', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores, name=f'Talmon f={f0_2} Hz', marker_color='green', showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores, name=f'Score f={f0_1} Hz', marker_color='blue'), row=5, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores, name=f'Score f={f0_2} Hz', marker_color='green'), row=5, col=1)


fig.update_xaxes(title_text="Time (s)", row=5, col=1)
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Distance (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
fig.update_yaxes(title_text="Score", row=5, col=1)

fig.update_layout(height=800, width=1200, title_text=f"Scooter experiment 1000m, scores for f0 = {f0_1}, {f0_2} Hz")
fig.show(renderer="browser")

In [10]:
_data = data[3000*fs:3060*fs]
fft = np.fft.rfft(_data)[20:]
freqs = np.fft.rfftfreq(len(_data), d=1/fs)[20:]

fig = go.Figure(data=go.Scatter(x=freqs, y=np.abs(fft)))
fig.show()

In [12]:
def s2g_band(x, fs, f0, BW, s2g_quantization_levels, phase_hop, s2g_mode):
    phase = band_phase_complex_demod(x, fs, f0, BW)
    phase_subsampled = phase[::phase_hop]
    transitions = ut.get_s2g(phase_subsampled, s2g_quantization_levels)
    K = ut.get_K(transitions, s2g_mode)
    return K

In [37]:
_data1 = data[2400*fs:2460*fs]
phase_hop = 16000
phase11 = band_phase_complex_demod(x=_data1, fs=fs, f0=f0_1, BW=10)
phase_subsampled11 = np.array([np.average(phase11[i:i+phase_hop]) for i in range(0, len(phase11), phase_hop)])
# phase_subsampled11 = phase11[::phase_hop]
phase_subsampled11 / 2

array([  0.40164568,  -1.0640235 ,   5.04039507,   1.7415036 ,
        -4.41639255,   3.12728957,   4.064242  ,   7.43001341,
        11.63711608,   9.96922727,   4.22661664,  -2.07844972,
       -11.37788977, -13.41721459,  -8.39376554, -12.31718506,
       -14.03071559, -18.2875283 , -23.19709914, -28.50584481,
       -30.41607305, -31.59725606, -38.86141321, -51.41356261,
       -62.26990603, -64.65346323, -67.28515774, -68.63193429,
       -70.2697254 , -62.49233313, -54.49947727, -52.33095371,
       -45.55118344, -41.3988999 , -41.75119622, -45.04824392,
       -44.62379453, -47.54432147, -48.93925731, -49.92120064,
       -45.58813306, -45.22516001, -47.07630108, -41.61356749,
       -33.99633954, -27.47150033, -18.14404042, -14.11230212,
       -11.0805527 ,  -0.95561711,   3.52070889,   1.80073859,
         4.07618378,   6.67123914,   7.69629533,   8.57436849,
        12.71707955,  18.06423326,  14.8567942 ,  15.32582667,
        15.82472336,  14.33581855,   5.60050121,   5.95

In [45]:
f0_1 = 631
f0_2 = 859
phase_hop = 16000
s2g_quantization_levels = 5
s2g_mode = 'wasserstein'

_data1 = data[2400*fs:2460*fs]
phase11 = band_phase_complex_demod(x=_data1, fs=fs, f0=f0_1, BW=10)
phase_subsampled11 = np.array([np.average(phase11[i:i+phase_hop]) for i in range(0, len(phase11), phase_hop)])
transitions11 = ut.get_s2g(phase_subsampled11, s2g_quantization_levels)
K11 = ut.get_K(transitions11, s2g_mode)

phase12 = band_phase_complex_demod(x=_data1, fs=fs, f0=f0_2, BW=10)
phase_subsampled12 = np.array([np.average(phase12[i:i+phase_hop]) for i in range(0, len(phase11), phase_hop)])
transitions12 = ut.get_s2g(phase_subsampled12, s2g_quantization_levels)
K12 = ut.get_K(transitions12, s2g_mode)

_data2 = data[3000*fs:3060*fs]
phase21 = band_phase_complex_demod(x=_data2, fs=fs, f0=f0_1, BW=10)
phase_subsampled21 = np.array([np.average(phase21[i:i+phase_hop]) for i in range(0, len(phase11), phase_hop)])
transitions21 = ut.get_s2g(phase_subsampled21, s2g_quantization_levels)
K21 = ut.get_K(transitions21, s2g_mode)

phase22 = band_phase_complex_demod(x=_data2, fs=fs, f0=f0_2, BW=10)
phase_subsampled22 = np.array([np.average(phase22[i:i+phase_hop]) for i in range(0, len(phase11), phase_hop)])
transitions22 = ut.get_s2g(phase_subsampled22, s2g_quantization_levels)
K22 = ut.get_K(transitions22, s2g_mode)

fig = make_subplots(rows=2, cols=2, subplot_titles=[f"T = {f0_1}, D: {f0_1}, K: {K11:.2f}", f"T = {f0_1}, D: {f0_2}, K: {K12:.2f}", f"T = {f0_2}, D: {f0_1}, K: {K21:.2f}", f"T = {f0_2}, D: {f0_2}, K: {K22:.2f}"])
fig.add_trace(go.Heatmap(z=transitions11, x=np.arange(len(transitions11)), y=np.arange(len(transitions11)), showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=transitions12, x=np.arange(len(transitions12)), y=np.arange(len(transitions12)), showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=transitions21, x=np.arange(len(transitions21)), y=np.arange(len(transitions21)), showscale=False), row=2, col=1)
fig.add_trace(go.Heatmap(z=transitions22, x=np.arange(len(transitions22)), y=np.arange(len(transitions22)), showscale=False), row=2, col=2)
fig.update_layout(height=1000, width=1000)
fig.show()

In [ ]:
# sanity - check spectral power + K only for f0 = 631, 859
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f0_1_bd_score = []
f0_2_bd_score = []

f0_1_s2g_l_scores = []
f0_2_s2g_l_scores = []

f0_1_s2g_w_scores = []
f0_2_s2g_w_scores = []

f0_1_td_scores = []
f0_2_td_scores = []

f0_1 = 631  # Hz
f0_2 = 851  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    pxx, F_pxx = welch_detector.get_feature_vector(_data)
    f0_1_pxx = pxx[np.where(F_pxx >= f0_1)[0][0]]
    f0_2_pxx = pxx[np.where(F_pxx >= f0_2)[0][0]]
    f0_1_bd_score.append(f0_1_pxx)
    f0_2_bd_score.append(f0_2_pxx)

    # K, F_K = s2g_detector.get_feature_vector(_data)
    # f0_1_K_pxx = K[np.where(F_K >= f0_1)[0][0]]
    # f0_2_K_pxx = K[np.where(F_K >= f0_2)[0][0]]
    # f0_1_s2g_scores.append(f0_1_K_pxx)
    # f0_2_s2g_scores.append(f0_2_K_pxx)
    K1 = s2g_band(x=_data, fs=fs, f0=f0_1, BW=10, s2g_quantization_levels=5, phase_hop=1, s2g_mode='wasserstein')
    K2 = s2g_band(x=_data, fs=fs, f0=f0_2, BW=10, s2g_quantization_levels=5, phase_hop=1, s2g_mode='wasserstein')
    f0_1_s2g_w_scores.append(K1)
    f0_2_s2g_w_scores.append(K2)
    

    td_scores, td_F = talmon_detector.get_feature_vector(_data)
    f0_1_td_pxx = td_scores[np.where(td_F >= f0_1)[0][0]]
    f0_2_td_pxx = td_scores[np.where(td_F >= f0_2)[0][0]]
    f0_1_td_scores.append(f0_1_td_pxx)
    f0_2_td_scores.append(f0_2_td_pxx)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")
